In [ ]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

In [ ]:
import sys
import os
working_path = '???'
sys.path.append(working_path)
os.chdir(working_path)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 8

DATA_NAME = 'cats_dogs_500/'
class_list = ['cats', 'dogs']

DATA_NAME_PATH = working_path +  'data/' + DATA_NAME
DATASET_PATH = DATA_NAME_PATH + 'dataset/'

### Load Dataset (or equivalent)

In [ ]:
# Images finally saved in folders with the following patterns
# data_path/
#     train/
#        class0/
#           classs0_0.jpg
#           ...
#           classs0_k.jpg
#       class1/
#         classs1_0.jpg
#         ...
#         classs1_k.jpg
#       ...
#       classN/
#         classsN_0.jpg
#         ...
#         classsN_k.jpg
#     validation/
#        class0/
#        ...
#        classN/
#     test/
#        class0/
#        ...
#        classN

In [ ]:
import tensorflow as tf

In [ ]:
TRAIN_PATH = DATASET_PATH + 'train/'
print(TRAIN_PATH)
train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_PATH,
                                                      image_size=(IMG_SIZE,IMG_SIZE),
                                                      batch_size=BATCH_SIZE,
                                                      shuffle=True,
                                                      seed=42
                                                      )
class_names = train_ds.class_names
print(class_names)

In [ ]:
VAL_PATH = DATASET_PATH + 'validation/'
print(VAL_PATH)
val_ds = tf.keras.utils.image_dataset_from_directory(VAL_PATH,
                                                     image_size=(IMG_SIZE,IMG_SIZE),
                                                     batch_size=BATCH_SIZE,
                                                     shuffle=False,
                                                    )

TEST_PATH = DATASET_PATH + 'test/'
print(TEST_PATH)
test_ds = tf.keras.utils.image_dataset_from_directory(TEST_PATH,
                                                      image_size=(IMG_SIZE,IMG_SIZE),
                                                      batch_size=BATCH_SIZE,
                                                      shuffle=False   # important for evaluation
                                                      )

In [ ]:
# Display a few images and labels
import matplotlib.pyplot as plt
%matplotlib inline

class_names = train_ds.class_names

plt.figure(figsize=(15,10))
for images, labels in train_ds.take(1):   # take one batch
    for i in range(min(15, images.shape[0])):
        ax = plt.subplot(3,5,i+1)

        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()

### Load Existing Deep Learning Model

In [ ]:
import tensorflow as tf

In [ ]:
# [UPDATED]
# Uncommment pre-trained model of your choice

base_model = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                                include_top=False,
                                                weights="imagenet"
                                                )

'''
base_model = tf.keras.applications.VGG16(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                         include_top=False,
                                         weights="imagenet"
                                        )
'''

'''
base_model = tf.keras.applications.ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                            include_top=False,
                                            weights="imagenet"
                                          )
'''

In [ ]:
for layer in base_model.layers:
    print(layer.name, layer.output.shape)

In [ ]:
base_model.summary()

### Feature Extraction: Construct Your Model

In [ ]:
from tensorflow.keras.models import  Sequential, Model, load_model

final_model = tf.keras.Sequential([base_model,
                            tf.keras.layers.GlobalAveragePooling2D(),
                            tf.keras.layers.BatchNormalization(),
                            tf.keras.layers.Dense(128, activation='relu'),
                            tf.keras.layers.Dropout(0.5),
                            tf.keras.layers.Dense(1, activation='sigmoid')                        # For binary class
                            ])

In [ ]:
# Compile Model

final_model.layers[0].trainable = False
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                    loss='binary_crossentropy',
                    metrics = ["accuracy"]
)

In [ ]:
for layer in final_model.layers:
    print(layer.name, layer.output.shape)

In [ ]:
final_model.summary()

In [ ]:
# For resizing images that are not 224 x 224
# X_train_resized = tf.image.resize(X_train, IMG_SIZE)
# X_val_resized = tf.image.resize(X_val, IMG_SIZE)
# X_test_resized = tf.image.resize(X_test, IMG_SIZE)

# print(f"Resized training images shape: {X_train_resized.shape}")
# print(f"Resized validation images shape: {X_val_resized.shape}")
# print(f"Resized validation images shape: {X_test_resized.shape}")

In [ ]:
# Uncomment below Data Augmentation
'''
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),  # flip horizontally
    tf.keras.layers.RandomRotation(0.2),       # 0.2 = rotate +/- 20% of 360 degress = +/- 72
    tf.keras.layers.RandomZoom(0.2)            # 0.2 = 80%-120% of original size
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
test_ds = test_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
'''

In [ ]:
# [UPDATED]
# Uncommment proprocess according to your pre-trained model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
# from tensorflow.keras.applications.vgg16 import preprocess_input
# from tensorflow.keras.applications.resnet50 import preprocess_input

train_ds_processed = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds_processed   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds_processed  = test_ds.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
# stops model training when the validation performance stops improving.
earlystopping = tf.keras.callbacks.EarlyStopping(patience=2,          # how many epochs the model waits before stopping.
                                                 monitor="val_loss",  # After every epoch, the model checks validation loss
                                                 restore_best_weights=True  # the model restores the weights from the best epoch.
                                                 )
history_model = final_model.fit(train_ds_processed,
                                epochs=10,                # Maximum number of training cycles.
                                batch_size=BATCH_SIZE,    # #training images that the model processes before updating its weights once.
                                validation_data=val_ds_processed,   # Use this dataset to compute validation loss
                                callbacks=[earlystopping]
                                )



In [ ]:
final_model.save(DATA_NAME_PATH + 'transfer_learning_model.keras')

In [ ]:
# [UPDATED]
# To load model
# model = tf.keras.models.load_model(DATA_NAME_PATH + 'transfer_learning_model.keras')

In [ ]:
# Plot the learning curves
import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))
plt.subplot(121)
plt.subplot(121)
try:
    plt.plot(history_model.history['accuracy'])
    plt.plot(history_model.history['val_accuracy'])
except KeyError:
    plt.plot(history_model.history['acc'])
    plt.plot(history_model.history['val_acc'])
plt.title('Accuracy vs. epochs')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='lower right')
plt.subplot(122)
plt.plot(history_model.history['loss'])
plt.plot(history_model.history['val_loss'])
plt.title('Loss vs. epochs')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='upper right')
plt.show()

In [ ]:
new_model_test_loss, new_model_test_acc = final_model.evaluate(test_ds_processed, verbose=0)
print("Test loss: {}".format(new_model_test_loss))
print("Test accuracy: {}".format(new_model_test_acc))

In [ ]:
prediction_probs = final_model.predict(test_ds_processed)
prediction_probs.flatten()

In [ ]:
y_predicted = (prediction_probs > 0.5).astype(int).flatten()
y_predicted

In [ ]:
import numpy as np

predicted_classes = [class_names[x] for x in y_predicted]

y_test = []
for images, labels in test_ds_processed:
   y_test.extend(labels.numpy())

y_test = np.array(y_test)
true_classes = [class_names[x] for x in y_test]

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Actual": true_classes,
    "Predicted": predicted_classes,
})
results_df.head()

In [ ]:
results_df["Correct"] = results_df["Actual"] == results_df["Predicted"]
num_correct = results_df["Correct"].sum()
print("Total Predictons: ", len(results_df), ", Correct predictions:", num_correct, ' Correct Percent: ', num_correct/len(results_df))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_classes, predicted_classes)
print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predicted, target_names=class_names))

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

y_test = np.concatenate([y.numpy() for x, y in test_ds_processed], axis=0)
y_prob = final_model.predict(test_ds_processed).flatten()

roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
class_names = np.array(class_list)

X_images = []
for images, labels in test_ds_processed:
    X_images.extend(images.numpy())
X_images = np.array(X_images)


plt.figure(figsize=(15,10))

# Random sample of test images
inx = np.random.choice(X_images.shape[0], 15, replace=False)

for n, i in enumerate(inx):
    ax = plt.subplot(3,5,n+1)
    img = (X_images[i] + 1) / 2
    plt.imshow(img)
    #plt.imshow(X_test[i] * 255)

    true_label = true_classes[i]
    pred_label = predicted_classes[i]

    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis('off')

plt.tight_layout()